In [1]:
pip install -q --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 55.3 MB/s eta 0:00:00


In [2]:
#Install Libraries
!pip install -q transformers datasets evaluate torchaudio accelerate jiwer peft torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.6 MB/s eta 0:00:00


In [3]:
#Load Dataset
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Cara Manual (Multi-Model & Multi-File)
import torch
import librosa
import os
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, display, HTML

# 1. Konfigurasi Perangkat
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan perangkat: {device}")

# 2. Daftar Model yang Ingin Dites
models_to_test = {
    "Minds14-Model": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/whisper-asr-minds14-Augm-epoch10-FINAL",
    "Standard-Model": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/whisper-Augm-epoch10-EarlyStoper-FINAL",
    "Small-Base": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/WhisperSmall-Augm-epoch10-EarlyStoper-FINAL",
    "Small-DoRA": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/WhisperSmallDORA-Augm-epoch10-EarlyStoper-FINAL",
    "Small-LoRA": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/best_whisper_lora"
}

folder_audio = "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Data-Test"

# 3. List semua file .wav
audio_files = [f for f in os.listdir(folder_audio) if f.endswith('.wav')]
audio_files.sort()

print(f"Memproses {len(audio_files)} file dengan {len(models_to_test)} model...\n")

# 4. Iterasi File Audio
for file_name in audio_files:
    path_lengkap = os.path.join(folder_audio, file_name)

    # Load audio sekali saja untuk semua model
    audio_input, sr = librosa.load(path_lengkap, sr=16000)

    # Tampilan Header Audio
    display(HTML(f"<hr><h3 style='color: #1a73e8;'>File: {file_name}</h3>"))
    display(Audio(audio_input, rate=sr))

    # 5. Iterasi Perbandingan Model
    for model_label, model_path in models_to_test.items():
        try:
            # Load Model & Processor secara bergantian
            model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
            processor = WhisperProcessor.from_pretrained(model_path)

            # Preprocessing & Inference
            input_features = processor(audio_input, sampling_rate=sr, return_tensors="pt").input_features.to(device)

            with torch.no_grad():
                predicted_ids = model.generate(input_features)

            transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]

            # Tampilkan Hasil per Model
            print(f"[{model_label}] : {transcription}")

            # Bersihkan memori agar RAM/VRAM tidak penuh
            del model
            del processor
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"Gagal memproses {model_label}: {e}")

    print("\n")

Menggunakan perangkat: cuda
Memproses 2 file dengan 5 model...



Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

[Minds14-Model] :  I have to upgrade my card to international payment I use Mastercard or Visa which is more suitable


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

[Standard-Model] :  I have to upgrade my card to international payment. Shall I choose Mastercard or Visa which is more suitable


...

WhisperForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/WhisperSmallDORA-Augm-epoch10-EarlyStoper-FINAL
Key                                                                                    | Status     | 
---------------------------------------------------------------------------------------+------------+-
model.decoder.layers.{0...11}.encoder_attn.v_proj.lora_magnitude_vector.default        | UNEXPECTED | 
model.encoder.layers.{0...11}.self_attn.q_proj.lora_magnitude_vector.default           | UNEXPECTED | 
model.encoder.layers.{0...11}.self_attn.v_proj.lora_magnitude_vector.default           | UNEXPECTED | 
model.decoder.layers.{0...11}.self_attn.v_proj.lora_magnitude_vector.default           | UNEXPECTED | 
model.decoder.layers.{0...11}.encoder_attn.q_proj.lora_magnitude_vector.default        | UNEXPECTED | 
model.decoder.layers.{0...11}.self_attn.q_proj.lora_magnitude_vector.default           | UNEXPECTED | 
model.encoder.layer

[Small-DoRA] :  this is the sound test for project 3 voice recognition


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

[Small-LoRA] :  this is the sound test for project 3 voice recognition




In [9]:
# Cara Manual (Multi-Model) - Transcribe & Translate Indonesia - MultiFile
import torch
import librosa
import os
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, display, HTML

# 1. Konfigurasi Perangkat
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan perangkat: {device}")

# 2. Daftar Model yang Ingin Dibandingkan
models_to_test = {
    "Minds14-Model": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/whisper-asr-minds14-Augm-epoch10-FINAL",
    "Standard-Model": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/whisper-Augm-epoch10-EarlyStoper-FINAL",
    "Small-Base": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/WhisperSmall-Augm-epoch10-EarlyStoper-FINAL",
    "Small-DoRA": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/WhisperSmallDORA-Augm-epoch10-EarlyStoper-FINAL",
    "Small-LoRA": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/best_whisper_lora"
}

folder_audio = "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Data-Test-Indo"

# 3. List semua file .wav
audio_files = [f for f in os.listdir(folder_audio) if f.endswith('.wav')]
audio_files.sort()

print(f"Memproses {len(audio_files)} file dengan {len(models_to_test)} model...\n")

# 4. Iterasi File Audio
for file_name in audio_files:
    path_lengkap = os.path.join(folder_audio, file_name)
    audio_input, sr = librosa.load(path_lengkap, sr=16000)

    # Tampilan Visual Audio
    display(HTML(f"<hr><h2 style='color: #d32f2f;'>File Audio: {file_name}</h2>"))
    display(Audio(audio_input, rate=sr))

    # 5. Iterasi Perbandingan Model
    for model_label, model_path in models_to_test.items():
        print(f"\n--- MENGGUNAKAN MODEL: {model_label} ---")

        try:
            # Load Model & Processor
            model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
            processor = WhisperProcessor.from_pretrained(model_path)

            input_features = processor(audio_input, sampling_rate=sr, return_tensors="pt").input_features.to(device)

            with torch.no_grad():
                # A. Transkripsi (Indonesia ke Indonesia)
                forced_ids_transcribe = processor.get_decoder_prompt_ids(language="indonesian", task="transcribe")
                predicted_ids_transcribe = model.generate(input_features, forced_decoder_ids=forced_ids_transcribe)

                # B. Translasi (Indonesia ke Inggris)
                forced_ids_translate = processor.get_decoder_prompt_ids(language="indonesian", task="translate")
                predicted_ids_translate = model.generate(input_features, forced_decoder_ids=forced_ids_translate)

                # C. Default
                predicted_ids_default = model.generate(input_features)

            # Decode Hasil
            trans_indo = processor.batch_decode(predicted_ids_transcribe, skip_special_tokens=True)[0]
            trans_eng  = processor.batch_decode(predicted_ids_translate, skip_special_tokens=True)[0]
            trans_def  = processor.batch_decode(predicted_ids_default, skip_special_tokens=True)[0]

            # Tampilan Hasil
            print(f"  [INDO-ASLI] : {trans_indo}")
            print(f"  [TRANSLATE] : {trans_eng}")
            print(f"  [DEFAULT]   : {trans_def}")

            # Bersihkan Memori
            del model
            del processor
            torch.cuda.empty_cache()

        except Exception as e:
            print(f"Error pada model {model_label}: {e}")

    print("\n" + "="*60)

Menggunakan perangkat: cuda
Memproses 1 file dengan 5 model...




--- MENGGUNAKAN MODEL: Minds14-Model ---


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

  [INDO-ASLI] :  ini adalah suara test untuk projek 3 untuk kembali kembali ke lampuk bi
  [TRANSLATE] :  this is a test test for project 3 for recognition recognition
  [DEFAULT]   :  this is a test for project 3 for recognition for B-shoot

--- MENGGUNAKAN MODEL: Standard-Model ---


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

...

WhisperForConditionalGeneration LOAD REPORT from: /content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/WhisperSmallDORA-Augm-epoch10-EarlyStoper-FINAL
Key                                                                                    | Status     | 
---------------------------------------------------------------------------------------+------------+-
model.decoder.layers.{0...11}.encoder_attn.v_proj.lora_magnitude_vector.default        | UNEXPECTED | 
model.encoder.layers.{0...11}.self_attn.q_proj.lora_magnitude_vector.default           | UNEXPECTED | 
model.encoder.layers.{0...11}.self_attn.v_proj.lora_magnitude_vector.default           | UNEXPECTED | 
model.decoder.layers.{0...11}.self_attn.v_proj.lora_magnitude_vector.default           | UNEXPECTED | 
model.decoder.layers.{0...11}.encoder_attn.q_proj.lora_magnitude_vector.default        | UNEXPECTED | 
model.decoder.layers.{0...11}.self_attn.q_proj.lora_magnitude_vector.default           | UNEXPECTED | 
model.encoder.layer

  [INDO-ASLI] :  ini adalah suara tes untuk proyek 3 voice recognition kelompok B
  [TRANSLATE] :  this is the sound test for project 3 voice recognition
  [DEFAULT]   :  this is the sound test for project 3 voice recognition

--- MENGGUNAKAN MODEL: Small-LoRA ---


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

  [INDO-ASLI] :  ini adalah suara tes untuk proyek 3 voice recognition kelompok B
  [TRANSLATE] :  this is the sound test for project 3 voice recognition
  [DEFAULT]   :  this is the sound test for project 3 voice recognition



In [13]:
import torch
import librosa
import os
from transformers import WhisperForConditionalGeneration, WhisperProcessor
from IPython.display import Audio, display, HTML

# 1. Konfigurasi Perangkat
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Menggunakan perangkat: {device}")

# 2. Definisi Daftar Model (Tambahkan path model Anda di sini)
models_to_test = {
    "Whisper-Minds14": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/whisper-asr-minds14-Augm-epoch10-FINAL",
    "Whisper-Standard": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/whisper-Augm-epoch10-EarlyStoper-FINAL",
    "Whisper-Small-Base": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/WhisperSmall-Augm-epoch10-EarlyStoper-FINAL",
    "Whisper-Small-DoRA": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/WhisperSmallDORA-Augm-epoch10-EarlyStoper-FINAL",
    "Small-LoRA": "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Model/best_whisper_lora"
}

folder_audio = "/content/drive/MyDrive/Project 03 - NLP - B -NalaPro/Data-Test"
audio_files = [f for f in os.listdir(folder_audio) if f.endswith('.wav')]
audio_files.sort()

# 3. Iterasi File Audio
for file_name in audio_files:
    path_lengkap = os.path.join(folder_audio, file_name)
    audio_input, sr = librosa.load(path_lengkap, sr=16000)

    display(HTML(f"<h3 style='color: #2e7d32;'>File: {file_name}</h3>"))
    display(Audio(audio_input, rate=sr))

    # 4. Iterasi Setiap Model untuk File yang sama
    for model_name, model_path in models_to_test.items():
        print(f"Sedang memproses dengan model: {model_name}...")

        # Load model dan processor (dilakukan di dalam loop agar bisa bergantian)
        # Catatan: Jika RAM tidak cukup, kita bisa menambahkan torch.cuda.empty_cache()
        model = WhisperForConditionalGeneration.from_pretrained(model_path).to(device)
        processor = WhisperProcessor.from_pretrained(model_path)

        input_features = processor(audio_input, sampling_rate=sr, return_tensors="pt").input_features.to(device)

        with torch.no_grad():
            transcribe_token_id = processor.tokenizer.convert_tokens_to_ids("<|transcribe|>")

            predicted_ids = model.generate(
                input_features,
                max_new_tokens=225,
                forced_decoder_ids=[(2, transcribe_token_id)],
                no_repeat_ngram_size=3,
                begin_suppress_tokens=[processor.tokenizer.pad_token_id],
                num_beams=5,
                do_sample=False
            )

        transcription = processor.batch_decode(predicted_ids, skip_special_tokens=True)[0]
        detected_lang_token = processor.tokenizer.decode(predicted_ids[0][1])

        # Tampilan Hasil Per Model
        print(f"[{model_name}]")
        print(f"Language: {detected_lang_token}")
        print(f"Result  : {transcription}")
        print("-" * 20)

        # Hapus model dari memori setelah selesai (opsional, untuk hemat VRAM)
        del model
        del processor
        torch.cuda.empty_cache()

    print("\n" + "="*60 + "\n")

Menggunakan perangkat: cuda


Sedang memproses dengan model: Whisper-Minds14...


Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Whisper-Minds14]
Language:  have
Result  :  I have to upgrade my card to international payment
--------------------
Sedang memproses dengan model: Whisper-Standard...


...

[Whisper-Small-DoRA]
Language:  is
Result  :  this is the sound test for project 3 voice recognition
--------------------
Sedang memproses dengan model: Small-LoRA...


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[Small-LoRA]
Language:  is
Result  :  this is the sound test for project 3 voice recognition
--------------------




# batas